# Probabilistic Programming: From BUGS to Stan and PyMC

## Historical problem

An important part of the computational revolution was not just inventing algorithms like Metropolis-Hastings or Hamiltonian Monte Carlo, but building software that made Bayesian modelling accessible. BUGS, WinBUGS, JAGS, Stan, and PyMC let users define a model directly and rely on the system to carry out inference.

This notebook revisits the simple Beta-Bernoulli example from the hand-coded MCMC notebook, but now the model is specified in PyMC rather than by manually writing the sampling logic.

In [ ]:
from pathlib import Path
import sys

import arviz as az
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
from scipy.stats import beta

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()

## Specify the model directly

In [ ]:
n = 100
s = 72
data = np.array([1] * s + [0] * (n - s))
alpha_prior = 2
beta_prior = 2

with pm.Model() as model:
    theta = pm.Beta("theta", alpha=alpha_prior, beta=beta_prior)
    pm.Bernoulli("obs", p=theta, observed=data)
    trace = pm.sample(1000, tune=1000, chains=2, cores=1, random_seed=123, progressbar=False, target_accept=0.9)

samples = trace.posterior["theta"].values.reshape(-1)
alpha_post = alpha_prior + s
beta_post = beta_prior + (n - s)

print(az.summary(trace, var_names=["theta"]))

In [ ]:
grid = np.linspace(0.001, 0.999, 500)
exact_density = beta.pdf(grid, alpha_post, beta_post)

running_mean = np.cumsum(samples) / np.arange(1, len(samples) + 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].hist(samples, bins=40, density=True, alpha=0.7, color="#4c78a8", label="PyMC posterior draws")
axes[0].plot(grid, exact_density, color="#d62728", lw=2, label="Exact Beta posterior")
axes[0].set_title("Probabilistic-programming posterior versus exact posterior")
axes[0].set_xlabel(r"$\theta$")
axes[0].set_ylabel("Density")
axes[0].legend()

axes[1].plot(samples[:1500], color="#54a24b", lw=1.0)
axes[1].axhline(samples.mean(), color="#d62728", ls="--", lw=1.5, label="Posterior mean")
axes[1].set_title("Trace plot for theta")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel(r"$\theta$")
axes[1].legend()

fig.tight_layout()
save_fig(fig, Path("figs") / "ppl_beta_bernoulli.png")
plt.show()

## Interpretation

The conceptual model is the same as in the hand-coded notebook, but the workflow is different:

- the user writes the probabilistic structure,
- the probabilistic-programming system manages inference,
- diagnostics and summaries come in a standard form.

That change in usability is one reason the later history of Bayesian computation includes software systems, not just algorithms.

## References

- Lunn, Thomas, Best, and Spiegelhalter (2000), *WinBUGS -- A Bayesian Modelling Framework*.
- Carpenter et al. (2017), *Stan: A Probabilistic Programming Language*.